**MultiQueryRetriever** is a LangChain tool that uses an LLM to rewrite a user's prompt into **multiple query variations**. It runs a vector search for each variation and combines the results.

* **Problem Solved:** Fixes poor vector search results caused by bad phrasing, missing keywords, or strict word matching.
* **How it Works:** Original Query $\rightarrow$ LLM generates 3–4 rephrased queries $\rightarrow$ Retrieves documents for all queries $\rightarrow$ Removes duplicates $\rightarrow$ Returns combined results.


* **Key Advantage:** Increases **recall** by capturing relevant documents that standard single-query searches miss.

In [22]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever 
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline  

In [ ]:
# Embedding model
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6218.78it/s]


In [9]:
# chat model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Create a Hugging Face text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

# 3. Wrap the pipeline in LangChain's HuggingFacePipeline wrapper
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4046.38it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [5]:
# Create faiss vector store
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding
)

In [ ]:
# create retrievers
similarity_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':5}
)

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs = {'k':5}),
    llm = llm
)

In [17]:
# Query
query = "How to improve energy levels and maintain balance?"

In [18]:
# retriever result
similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiquery_retriever.invoke(query)

In [21]:
for i, doc in enumerate(similarity_result):
    print(f'\n--- Result {i+1} ---')
    print(doc.page_content)
print("*"*100)

for i,doc in enumerate(multiquery_result[:5]):
    print(f'\n--- Result {i+1} ---')
    print(doc.page_content)
print("*"*100)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
****************************************************************************************************

--- Result 1 ---
Python balances readability with power, making it a popular system design language.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.

--- Result 4 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
**********************